In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
def multi_col_apply(func):
    def wrapper(df, cols=None):
        if cols is None:
            cols = df.columns
        for c in cols:
            df = func(df, c)
        return df
    return wrapper

def apply_transformations(df, transformations):
    for func, kwargs in transformations:
        df = func(df, **kwargs)
    return df

In [0]:
@multi_col_apply
def trimming(df, col):
    return df.withColumn(col, F.trim(F.col(col)))

@multi_col_apply
def uppercase(df, col):
    return df.withColumn(col, F.upper(F.col(col)))

@multi_col_apply
def lowercase(df, col):
    return df.withColumn(col, F.lower(F.col(col)))

@multi_col_apply
def capitalize(df, col):
    return df.withColumn(col, F.initcap(F.col(col)))

def mobile_num_cleanup(df, col):
    new_col_name = col + '_cleaned'
    return (
        df
        .withColumn(new_col_name, F.regexp_replace(F.col(col), r"^\+\d{2}[\s\-\(\)]*|[\s\-\(\)]", ""))
        .withColumn(new_col_name, F.substring(new_col_name, -10,10))
        .filter(F.length(F.col(new_col_name)) == 10)
    )

def handle_duplicate(df, col1, col2):
    return (
        df
        .withColumn("rn", F.rank().over(Window.partitionBy(F.col(f"{col1}")).orderBy(F.col(f"{col2}"))))
        .filter(F.col("rn")==1)
        .drop("rn")
    )

In [0]:
def unify_customers_data(df_offline_customers, df_online_customers):
    online_cust_transformations = [
        (mobile_num_cleanup, {"col": "mobile_number"}),
        (trimming, {}),
        (capitalize, {"cols": ["full_name", "city"]}),
        (lowercase, {"cols": ["email"]}),
        (uppercase, {"cols": ["customer_id", "state"]}),
        (handle_duplicate, {"col1": "mobile_number_cleaned", "col2": "customer_id"})
    ]

    df_online_transformed = apply_transformations(df_online_customers, online_cust_transformations)

    offline_cust_transformations = [
        (mobile_num_cleanup, {"col": "phone"}),                          
        (trimming, {}),
        (capitalize, {"cols": ["name", "city"]}),
        (lowercase, {"cols": ["email"]}),
        (uppercase, {"cols": ["cust_id", "phone"]}),
        (handle_duplicate, {"col1": "phone_cleaned", "col2": "cust_id"})
    ]

    df_offline_transformed = apply_transformations(df_offline_customers, offline_cust_transformations)

    df_common_customers = (
    df_offline_transformed
    .join(df_online_transformed, df_offline_transformed["phone_cleaned"]==df_online_transformed["mobile_number_cleaned"], "inner")
    .select(F.coalesce(F.col("full_name"), F.col("name")).cast("string").alias("cust_name"),
            F.coalesce(df_online_transformed["email"], df_offline_transformed["email"]).cast("string").alias("cust_email"),
            F.col("mobile_number_cleaned").cast("string").alias("cust_mobile"),           
            F.coalesce(df_online_transformed["city"], df_offline_transformed["city"]).cast("string").alias("cust_city"),
            F.coalesce(df_online_transformed["state"], df_offline_transformed["state"]).cast("string").alias("cust_state")
            )
    )

    df_online_only_customers = (
    df_online_transformed
    .join(df_offline_transformed, df_online_transformed["mobile_number_cleaned"] == df_offline_transformed["phone_cleaned"], "left_anti")
    .select(F.col("full_name").cast("string").alias("cust_name"),
            df_online_transformed["email"].cast("string").alias("cust_email"),
            F.col("mobile_number_cleaned").cast("string").alias("cust_mobile"),           
            df_online_transformed["city"].cast("string").alias("cust_city"),
            df_online_transformed["state"].cast("string").alias("cust_state")
            )
    )

    df_offline_only_customers = (
    df_offline_transformed
    .join(df_online_transformed, df_online_transformed["mobile_number_cleaned"] == df_offline_transformed["phone_cleaned"], "left_anti")
    .select(F.col("name").cast("string").alias("cust_name"),
            df_offline_transformed["email"].cast("string").alias("cust_email"),
            F.col("phone_cleaned").cast("string").alias("cust_mobile"),           
            df_offline_transformed["city"].cast("string").alias("cust_city"),
            df_offline_transformed["state"].cast("string").alias("cust_state")
            )
    )

    df_customers_unified = df_common_customers.union(df_online_only_customers).union(df_offline_only_customers)

    return df_customers_unified

In [0]:
def unify_products_data(df_offline_products, df_online_products):
    online_products_transformations = [
        (trimming, {}),
        (capitalize, {"cols": ["product_name", "category", "brand"]}),
        (uppercase, {"cols": ["sku_id"]})
    ]

    df_online_transformed = apply_transformations(df_online_products, online_products_transformations)

    offline_products_transformations = [
        (trimming, {}),
        (capitalize, {"cols": ["name", "category_name", "brand_name"]}),
        (uppercase, {"cols": ["sku"]}),
    ]

    df_offline_transformed = apply_transformations(df_offline_products, offline_products_transformations)

    df_common_products = (
        df_offline_transformed
        .join(df_online_transformed, df_offline_transformed["sku"]==df_online_transformed["sku_id"], "inner")
        .select(F.col("sku_id").cast("string").alias("product_id"),           
                F.coalesce(F.col("product_name"), F.col("name")).cast("string").alias("product_name"),
                F.coalesce(F.col("category"), F.col("category_name")).cast("string").alias("product_category"),
                F.coalesce(F.col("brand"), F.col("brand_name")).cast("string").alias("product_brand"),
                F.coalesce(F.col("price"), F.col("mrp_price")).cast("float").alias("product_price")
                )
    )

    df_online_only_products = (
        df_online_transformed
        .join(df_offline_transformed, df_online_transformed["sku_id"] == df_offline_transformed["sku"], "left_anti")
        .select(F.col("sku_id").cast("string").alias("product_id"),           
                F.col("product_name").cast("string").alias("product_name"),
                F.col("category").cast("string").alias("product_category"),
                F.col("brand").cast("string").alias("product_brand"),
                F.col("price").cast("float").alias("product_price")
                )
    )

    df_offline_only_products = (
        df_offline_transformed
        .join(df_online_transformed, df_online_transformed["sku_id"] == df_offline_transformed["sku"], "left_anti")
        .select(F.col("sku").cast("string").alias("product_id"),           
                F.col("name").cast("string").alias("product_name"),
                F.col("category_name").cast("string").alias("product_category"),
                F.col("brand_name").cast("string").alias("product_brand"),
                F.col("mrp_price").cast("float").alias("product_price")
                )
    )

    df_products_unified = df_common_products.union(df_online_only_products).union(df_offline_only_products)

    return df_products_unified


In [0]:
df_offline_customers = spark.table("retail_data.bronze.offline_customers")
df_online_customers = spark.table("retail_data.bronze.online_customers")

In [0]:
unified_customers = unify_customers_data(df_offline_customers, df_online_customers)
unified_customers.write.mode("overwrite").saveAsTable('retail_data.silver.customers_unified')

In [0]:
df_offline_products = spark.table("retail_data.bronze.offline_products")
df_online_products = spark.table("retail_data.bronze.online_products")

In [0]:
unified_products = unify_products_data(df_offline_products, df_online_products)
unified_products.write.mode("overwrite").saveAsTable('retail_data.silver.products_unified')